In [ ]:
!pip install -q transformers datasets evaluate rouge-score
!pip install -q bitsandbytes
!pip install evaluate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.8 MB/s eta 0:00:00


In [ ]:
import torch
import bitsandbytes as bnb
print("bitsandbytes loaded")


bitsandbytes loaded


In [ ]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 50.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import transformers
print(transformers.__version__)


5.2.0


In [ ]:
import torch
import re
import pandas as pd

from datasets import Dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)

from peft import LoraConfig, get_peft_model
import evaluate


In [ ]:
df = pd.read_csv("Reviews.csv")

df = df[['Text', 'Summary', 'Score']]   # ← keep Score

df.rename(
    columns={'Text': 'reviewText', 'Summary': 'summary'},
    inplace=True
)

df.dropna(inplace=True)

df = df.sample(n=10000, random_state=42).reset_index(drop=True)


In [ ]:
print(df.head())
print(df.columns)


                                          reviewText  \
0  These are actually very tasty.  Pure potatoes ...   
1  I realize that taste is a matter of personal p...   
2  This is one of my Favorite cup of soup choices...   
3  If you like the classic taste of a good margar...   
4  I was willing to give this a chance even after...   

                                           summary  Score  
0                                    I like these!      4  
1                 Good but subjectively not 5 star      4  
2         Lipton Cup A Soup, Spring Vegetable.4 oz      5  
3  Suited to its purpose, if not quite its goal...      4  
4                               Tastes artificial!      2  
Index(['reviewText', 'summary', 'Score'], dtype='object')


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['reviewText'] = df['reviewText'].apply(clean_text)
df['summary'] = df['summary'].apply(clean_text)


In [ ]:
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2, seed=42)

train_ds = dataset["train"]
test_ds = dataset["test"]


In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [ ]:
MAX_INPUT = 256
MAX_TARGET = 64

def preprocess(batch):
    model_inputs = tokenizer(
        ["summarize: " + t for t in batch["reviewText"]],
        max_length=MAX_INPUT,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=batch["summary"],
        max_length=MAX_TARGET,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [ ]:
train_ds = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
test_ds = test_ds.map(preprocess, batched=True, remove_columns=test_ds.column_names)


Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = T5ForConditionalGeneration.from_pretrained(
    "t5-small",
    quantization_config=bnb_config,
    device_map="auto"
)


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM",
    target_modules=["q", "v"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 294,912 || all params: 60,801,536 || trainable%: 0.4850


In [ ]:
training_args = TrainingArguments(
    output_dir="./t5_qlora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=2,
    logging_steps=100,
    report_to="none",
    fp16=True
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds
)


In [ ]:
trainer.train()


Step,Training Loss
100,34.163601
200,3.266092
300,2.524504
400,2.134086
500,1.922042
600,1.751844
700,1.596692
800,1.671829
900,1.643648
1000,1.533262


TrainOutput(global_step=2000, training_loss=3.3962054290771486, metrics={'train_runtime': 1117.4057, 'train_samples_per_second': 14.319, 'train_steps_per_second': 1.79, 'total_flos': 1089982169088000.0, 'train_loss': 3.3962054290771486, 'epoch': 2.0})

In [ ]:
trainer.save_model("./t5_qlora_model")
tokenizer.save_pretrained("./t5_qlora_model")


('./t5_qlora_model/tokenizer_config.json', './t5_qlora_model/tokenizer.json')

In [ ]:
model.eval()

# Take 10 random reviews
sample_df = df.sample(n=10).reset_index(drop=True)

for i in range(10):
    review = sample_df.loc[i, "reviewText"]
    true_summary = sample_df.loc[i, "summary"]   # dataset summary

    inputs = tokenizer(
        "summarize: " + review,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=60,
            min_length=15,
            num_beams=6,
            length_penalty=1.2,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    generated_summary = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    print(f" REVIEW {i+1}")
    print("Review Text:")
    print(review[:500], "..." if len(review) > 500 else "")

    print("\nOriginal Summary (Dataset):")
    print(true_summary)

    print("\nModel Generated Summary:")
    print(generated_summary)
    print("=" * 80)


 REVIEW 1
Review Text:
i was pleasantly surprised when i took my first sip of "illy issimo caffe: italian espresso style coffee drink." i'm not sure what i expected, but it certainly wasn't the wonderful taste of real espresso coffee from a can. i'm a skeptic and proud of it! my only quibble, and this is a matter of personal taste, i found this coffee drink to be very sweet. i am not a sweet guy, no matter what my wife says. i simply don't drink my coffee with sugar (real or artificial), nor do i put milk/cream in it. ...

Original Summary (Dataset):
ain't she sweet...

Model Generated Summary:
i'm a skeptic and proud of it!
 REVIEW 2
Review Text:
the only thing i can honestly say that i like about the dog food is the convenient, reseal able bag. we had a stray appear in our neighborhood and although i'm not an avid animal lover, i also don't like to see animals suffer so we began feeding him. when offered this to try and review i had hoped it would be a chance to try a nutritious, who

In [ ]:
# store summaries
generated_data = []

for i in range(len(df)):
    review = df.loc[i, "reviewText"]

    inputs = tokenizer(
        "summarize: " + review,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=60
        )

    summary = tokenizer.decode(output[0], skip_special_tokens=True)

    generated_data.append(summary)

df["generated_summary"] = generated_data


In [ ]:
from transformers import BertTokenizer, BertModel

bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased")
bert_model.eval()

for param in bert_model.parameters():
    param.requires_grad = False


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def get_bert_embeddings_batch(texts, max_len=128):
    inputs = bert_tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = bert_model(**inputs)

    return outputs.last_hidden_state


In [ ]:
import torch
import torch.nn as nn

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim, num_heads, batch_first=True
        )
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + attn_out)
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)
        return x


In [ ]:
class BiGRUEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.bigru = nn.GRU(
            input_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, x):
        outputs, hidden = self.bigru(x)
        return outputs, hidden


In [ ]:
class Attention(nn.Module):
    def __init__(self, enc_dim, dec_dim):
        super().__init__()
        self.attn = nn.Linear(enc_dim + dec_dim, dec_dim)
        self.v = nn.Linear(dec_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        seq_len = encoder_outputs.size(1)
        hidden = hidden.unsqueeze(1).repeat(1, seq_len, 1)
        energy = torch.tanh(
            self.attn(torch.cat((hidden, encoder_outputs), dim=2))
        )
        attention = self.v(energy).squeeze(2)
        return torch.softmax(attention, dim=1)


In [ ]:
class GRUDecoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, hidden):
        output, _ = self.gru(x, hidden)
        return self.fc(output[:, -1, :])


In [ ]:
class SummaryRatingModel(nn.Module):
    def __init__(self, embed_dim, hidden_dim, num_classes):
        super().__init__()

        self.transformer = TransformerBlock(embed_dim, 4, 256)
        self.encoder = BiGRUEncoder(embed_dim, hidden_dim)
        self.attention = Attention(hidden_dim * 2, hidden_dim)
        self.decoder = GRUDecoder(hidden_dim * 2, hidden_dim, num_classes)

    def forward(self, x):
        x = self.transformer(x)
        enc_out, hidden = self.encoder(x)

        hidden = hidden[-1]
        attn_weights = self.attention(hidden, enc_out)
        context = torch.bmm(attn_weights.unsqueeze(1), enc_out)

        return self.decoder(context, hidden.unsqueeze(0))


In [ ]:
class SummaryDataset(Dataset):
    def __init__(self, summaries, labels):
        self.summaries = summaries
        self.labels = labels

    def __len__(self):
        return len(self.summaries)

    def __getitem__(self, idx):
        return self.summaries[idx], self.labels[idx]


In [ ]:
print(df.columns)


Index(['reviewText', 'summary', 'Score', 'generated_summary'], dtype='object')


In [ ]:
from torch.utils.data import Dataset

class SummaryDataset(Dataset):
    def __init__(self, summaries, labels):
        self.summaries = summaries          # list of strings
        self.labels = labels                # list of ints

    def __len__(self):
        return len(self.summaries)

    def __getitem__(self, idx):
        return self.summaries[idx], self.labels[idx]


In [ ]:
labels = (df["Score"].values - 1).tolist()


In [ ]:
from torch.utils.data import DataLoader

dataset = SummaryDataset(
    df["generated_summary"].astype(str).tolist(),
    labels
)

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True
)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

rating_model = SummaryRatingModel(
    embed_dim=768,
    hidden_dim=128,
    num_classes=5
).to(device)

optimizer = torch.optim.Adam(rating_model.parameters(), lr=1e-4)
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
for epoch in range(10):
    rating_model.train()
    total_loss = 0

    for summaries, labels in loader:
        optimizer.zero_grad()

        embeddings = get_bert_embeddings_batch(summaries).to(device)
        labels = labels.to(device).long()

        outputs = rating_model(embeddings)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss/len(loader):.4f}")

Epoch 1 Loss: 1.0685
Epoch 2 Loss: 1.0152
Epoch 3 Loss: 0.9730
Epoch 4 Loss: 0.9127
Epoch 5 Loss: 0.8347
Epoch 6 Loss: 0.7397
Epoch 7 Loss: 0.6421
Epoch 8 Loss: 0.5651
Epoch 9 Loss: 0.5182
Epoch 10 Loss: 0.4843


In [ ]:
rating_model.eval()

correct = 0
total = 0

with torch.no_grad():
    for summaries, labels in loader:
        embeddings = get_bert_embeddings_batch(summaries).to(device)
        labels = labels.to(device).long()

        outputs = rating_model(embeddings)
        preds = torch.argmax(outputs, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

overall_accuracy = correct / total
print(f"Overall Accuracy: {overall_accuracy * 100:.2f}%")

Overall Accuracy: 84.85%


In [ ]:
import evaluate

# load rouge metric
rouge = evaluate.load("rouge")

# convert dataframe columns to list
predictions = df["generated_summary"].tolist()
references  = df["summary"].tolist()

# compute rouge
results = rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=True
)

print("ROUGE Scores:")
for key, value in results.items():
    print(f"{key}: {value:.4f}")

ROUGE Scores:
rouge1: 0.1326
rouge2: 0.0350
rougeL: 0.1299
rougeLsum: 0.1299


In [ ]:
rating_model.eval()

import random

# take 10 random samples
sample_df = df.sample(n=10).reset_index(drop=True)

print("\n SAMPLE RATING PREDICTIONS \n")

for i in range(len(sample_df)):

    review_text = sample_df.loc[i, "reviewText"]
    summary = sample_df.loc[i, "generated_summary"]
    true_rating = sample_df.loc[i, "Score"]

    # Get BERT embedding of summary
    embedding = get_bert_embeddings_batch([summary]).to(device)

    with torch.no_grad():
        output = rating_model(embedding)
        pred_class = torch.argmax(output, dim=1).item()

    predicted_rating = pred_class + 1   # because labels were 0–4

    print(f"Review {i+1}")
    print("Review Text:", review_text[:200], ".." if len(review_text)>200 else "")
    print("Generated Summary:", summary)
    print("True Rating:", true_rating)
    print("Predicted Rating:", predicted_rating)
    print("-"*60)


 SAMPLE RATING PREDICTIONS 

Review 1
Review Text: fell for the taste on the very first bite...prefer the chewy texture over the other crunchy bars, which almost break my teeth off...like date syrup flavor shining through 
Generated Summary: chewy texture
True Rating: 5
Predicted Rating: 5
------------------------------------------------------------
Review 2
Review Text: non-ge - "ge" stands for genetic engineering and is also referred to as gmo (genetically modified organism). genetic engineering of food products uses gene splicing to overcome natural boundaries that ..
Generated Summary: non-ge
True Rating: 5
Predicted Rating: 5
------------------------------------------------------------
Review 3
Review Text: because i like sweet potatoe fries, i thought i would like these. but, i found them to lack flavor. there is just a bit of a bitter taste to them. they may be better for you than potato chips, but the ..
Generated Summary: i like sweet potatoe fries
True Rating: 2
Predicted Ra